# 🚀 POPE Benchmark Evaluation with ONLY (ICCV'25)
### Environment: Kaggle 2× NVIDIA T4 GPUs | BF16 Full Precision (No Quantization)

This notebook evaluates **POPE** (Random, Popular, Adversarial) hallucination benchmark using the **ONLY** intervention method on:
- **LLaVA-1.5-7B** (`llava-hf/llava-1.5-7b-hf`)
- **Qwen2-VL-7B-Instruct** (`Qwen/Qwen2-VL-7B-Instruct`)

**Key Technical Specifications:**
- **Hardware**: 2× NVIDIA T4 GPUs (32GB combined VRAM) via `device_map="auto"`.
- **Precision**: `torch.bfloat16` Full Precision (Strictly NO quantization / bitsandbytes).
- **Decoding**: Greedy decoding (`do_sample=False`, `temperature=0.0`), strictly `max_new_tokens=6`.
- **Prompting**: Suffix `" Please answer with yes or no."` for QwenVL; verbatim prompt for LLaVA.
- **Efficiency**: `--split all` loads model weights once into GPU memory, then evaluates all 3 splits.

In [ ]:
# ==============================================================================
# CELL 1: Environment Setup & Anti-Conflict Dependency Installation (MANDATORY)
# ==============================================================================
# 1. Gỡ bỏ torchaudio (dự án chỉ dùng Ảnh + Chữ, gỡ bỏ để tránh 100% xung đột CUDA mismatch)
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết (KHÔNG cài torch/torchvision để giữ nguyên driver CUDA của Kaggle)
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    huggingface_hub \
    pandas

# 3. DÒNG KIỂM TRA XÁC THỰC DEPENDENCY NGAY TẠI CELL 1:
import torch, transformers, accelerate, qwen_vl_utils, sentencepiece
from transformers import AutoProcessor, AutoTokenizer
print(f"✅ Dependency Verification PASSED! PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()}) | Transformers: {transformers.__version__}")

In [ ]:
# ==============================================================================
# CELL 2: HuggingFace Authentication (via Kaggle Secrets 'HF_TOKEN')
# ==============================================================================
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace Hub login successful with Kaggle Secret 'HF_TOKEN'!")
except Exception as e:
    print(f"ℹ️ Note: Could not authenticate via Kaggle Secrets ({e}).")
    print("   Public models (llava-hf/llava-1.5-7b-hf, Qwen/Qwen2-VL-7B-Instruct) can be loaded without login.")

In [ ]:
# ==============================================================================
# CELL 3: Clone Repository ONLY & Set Working Directory
# ==============================================================================
import os
import subprocess

REPO_URL = "https://github.com/ntmy12/ONLY.git"
REPO_DIR = "/kaggle/working/ONLY"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✅ Clone completed successfully!")
else:
    print(f"Repository already exists at {REPO_DIR}.")

%cd {REPO_DIR}
!pwd

In [ ]:
# ==============================================================================
# CELL 4: Hardware & GPU Environment Verification
# ==============================================================================
import torch

num_gpus = torch.cuda.device_count()
print(f"GPUs available: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    mem_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  GPU {i}: {name} ({mem_gb:.1f} GB VRAM)")

bf16_ok = torch.cuda.is_bf16_supported()
print(f"BF16 Hardware Support: {'✅ YES' if bf16_ok else '⚠️ Emulated/Limited'}")

In [ ]:
# ==============================================================================
# CELL 5: Auto-Detection Test for COCO val2014 & POPE Annotations
# ==============================================================================
import sys
sys.path.insert(0, "/kaggle/working/ONLY")
from eval_bench.pope_auto_detect import find_coco_images, find_or_download_pope_annotations

try:
    coco_dir = find_coco_images()
    print(f"✅ COCO val2014 images found: {coco_dir}")
except Exception as e:
    print(f"⚠️ {e}")

for split in ["random", "popular", "adversarial"]:
    pope_file = find_or_download_pope_annotations(split=split)
    print(f"✅ POPE {split} annotation: {pope_file}")

In [ ]:
# ==============================================================================
# CELL 6: Run POPE Benchmark (--split all, 2x T4 GPUs, BF16)
# ==============================================================================
# Select model: 'llava' or 'qwen2vl'
MODEL_CHOICE = "llava"  # change to 'qwen2vl' to evaluate Qwen2-VL

print(f"Starting POPE evaluation for {MODEL_CHOICE.upper()} with ONLY intervention...")

!python eval_bench/eval_pope.py \
    --model {MODEL_CHOICE} \
    --split all \
    --use_only True \
    --max_new_tokens 6 \
    --precision auto \
    --device_map auto \
    --out_path ./results \
    --run_dir ./results/{MODEL_CHOICE}_only_pope

In [ ]:
# ==============================================================================
# CELL 7: Summary Metrics Display & Result Verification
# ==============================================================================
import glob
import os
import pandas as pd

# Locate the most recent results summary CSV
summary_files = sorted(glob.glob("./results/*/summary_metrics.csv"), key=os.path.getmtime)
if summary_files:
    latest_csv = summary_files[-1]
    print(f"Displaying latest results from: {latest_csv}\n")
    df = pd.read_csv(latest_csv)
    pd.set_option('display.precision', 2)
    display(df)
else:
    print("No summary_metrics.csv found in ./results yet.")